In [9]:
!pip install -qU crewai[tools,agentops]==0.95.0

ERROR: Ignored the following yanked versions: 0.165.0, 1.10.0, 1.12.0, 1.14.0
ERROR: Ignored the following versions that require a different python version: 0.100.0 Requires-Python >=3.10,<3.13; 0.100.1 Requires-Python >=3.10,<3.13; 0.102.0 Requires-Python >=3.10,<3.13; 0.105.0 Requires-Python >=3.10,<3.13; 0.108.0 Requires-Python >=3.10,<3.13; 0.114.0 Requires-Python >=3.10,<3.13; 0.117.0 Requires-Python >=3.10,<3.13; 0.117.1 Requires-Python >=3.10,<3.13; 0.118.0 Requires-Python >=3.10,<3.13; 0.119.0 Requires-Python >=3.10,<3.13; 0.120.0 Requires-Python >=3.10,<3.13; 0.120.1 Requires-Python >=3.10,<3.13; 0.121.0 Requires-Python >=3.10,<3.13; 0.121.1 Requires-Python >=3.10,<3.13; 0.14.0 Requires-Python >=3.10,<=3.13; 0.14.0rc0 Requires-Python >=3.10,<3.12; 0.14.0rc1 Requires-Python >=3.10,<=3.13; 0.14.1 Requires-Python >=3.10,<=3.13; 0.14.3 Requires-Python >=3.10,<=3.13; 0.14.4 Requires-Python >=3.10,<=3.13; 0.16.0 Requires-Python >=3.10,<=3.13; 0.16.1 Requires-Python >=3.10,<=3.13; 0.

In [17]:
import sys
!{sys.executable} -m pip install agentops

  Using cached agentops-0.4.21-py3-none-any.whl.metadata (2.1 kB)
  Using cached opentelemetry_instrumentation-0.62b1-py3-none-any.whl.metadata (7.2 kB)
  Using cached ordered_set-4.1.0-py3-none-any.whl.metadata (5.3 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached psutil-7.0.0-cp37-abi3-win_amd64.whl.metadata (23 kB)
  Using cached termcolor-2.4.0-py3-none-any.whl.metadata (6.1 kB)
INFO: pip is looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
Using cached agentops-0.4.21-py3-none-any.whl (309 kB)
Using cached ordered_set-4.1.0-py3-none-any.whl (7.6 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached psutil-7.0.0-cp37-abi3-win_amd64.whl (244 kB)
Using cached term

In [1]:
from crewai import Agent, Task, Crew, Process, LLM
import agentops

import os 
from dotenv import load_dotenv

In [5]:
# Load the variables from .env into the system environment
load_dotenv()

agentops_key = os.getenv("AGENTOPS_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

# Initialize AgentOps
agentops.init(
    api_key = os.getenv("AGENTOPS_API_KEY"),
    default_tags=('crewai')
)


In [6]:
# create new repo for outputs 
output_dir = "./ai-agents-output"
os.makedirs(output_dir, exist_ok=True)

basic_llm = LLM(model="llama-3.3-70b-versatile",temperature=0)

## Setup Agents

In [10]:
search_queries_recommendation_agent = Agent(
    role = "Search Queries Recommendation Agent",
    goal = "\n".join([
        "to provide a list of suggested searh queries to be passed to the search engine.",
        "the queries must be varied and looking for specific items."
    ]),
    backstory = "The agent is designed to help in looking for product by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm = basic_llm,
    verbose = True,
)

search_queries_recommendation_task = Task(
    description= "\n".join([
        "Rankyx is looking to buy {product_name} at the best prices (value for a price strategy)",
        "the company target any of these websites to buy from : {websites_list}",
        "the company want to search all available products on the internet to be compared later in another stage",
        "The stores must sell the product in {contry_name}",
        "Generate only {no_keywords} queries",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),
    expected_output= "A JSON object containing a list of suggested search queries.",
    output_json = None,
)